In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Monitorizează sau anulează un job

Vizualizează o listă a sarcinilor tale de lucru pe [pagina Workloads](https://quantum.cloud.ibm.com/workloads).

## Vizualizează starea unui job
Accesează [tabelul Workloads](https://quantum.cloud.ibm.com/workloads) și verifică coloana Status pentru a vedea dacă un job s-a finalizat sau a eșuat.

## Vizualizează utilizarea rămasă
Accesează [tabelul Instances](https://quantum.cloud.ibm.com/instances) și selectează tab-ul asociat planului pentru care vrei să vizualizezi utilizarea rămasă. Sunt afișate timpul total utilizat și timpul total rămas din planul tău.

## Vizualizează metricile privind numărul de joburi și sarcini de lucru trimise
Accesează [pagina Analytics](https://quantum.cloud.ibm.com/analytics) pentru a vedea numărul total de joburi trimise, precum și numărul de sarcini de lucru de tip batch și session. Reține că poți vedea pagina Analytics doar pentru conturile pe care le deții sau le administrezi.

## Monitorizează un job
Folosește instanța jobului pentru a verifica starea acestuia sau pentru a prelua rezultatele, apelând comanda corespunzătoare:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Vizualizează rezultatele jobului imediat după finalizarea acestuia. Rezultatele jobului sunt disponibile după finalizarea sa. Prin urmare, job.result() este un apel blocant până la finalizarea jobului.               |
| job.job\_id()                 | Returnează ID-ul care identifică în mod unic acel job. Preluarea rezultatelor jobului la o dată ulterioară necesită ID-ul jobului. Prin urmare, este recomandat să salvezi ID-urile joburilor pe care ai putea dori să le recuperezi ulterior. |
| job.status()                  | Verifică starea jobului.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Recuperează un job trimis anterior. Acest apel necesită ID-ul jobului.                                                                                                                                      |

<span id="retrieve-later"></span>
## Recuperează rezultatele unui job la o dată ulterioară
Apelează `service.job(\<job\_id>)` pentru a recupera un job trimis anterior. Dacă nu ai ID-ul jobului sau dacă vrei să recuperezi mai multe joburi simultan, inclusiv joburi de pe QPU-uri (unități de procesare cuantică) retrase, apelează în schimb `service.jobs()` cu filtre opționale. Vezi [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` returnează și joburi rulate cu pachetul `qiskit-ibm-provider` deprecat. Joburile trimise cu pachetul mai vechi (de asemenea deprecat) `qiskit-ibmq-provider` nu mai sunt disponibile.

### Exemplu
Acest exemplu returnează cele mai recente 10 joburi runtime care au rulat pe `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Anulează un job
Poți anula un job din panoul de control IBM Quantum Platform, fie de pe pagina Workloads, fie de pe pagina de detalii a unei sarcini de lucru specifice. Pe pagina Workloads, fă clic pe meniul overflow de la sfârșitul rândului pentru acea sarcină de lucru și selectează Cancel. Dacă te afli pe pagina de detalii a unei sarcini de lucru specifice, folosește meniul dropdown Actions din partea de sus a paginii și selectează Cancel.

În Qiskit, folosește `job.cancel()` pentru a anula un job.

<span id="execution-spans"></span>
## Vizualizează intervalele de execuție Sampler
Rezultatele joburilor [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) executate în Qiskit Runtime conțin informații despre momentul execuției în metadatele lor.
Aceste informații pot fi folosite pentru a stabili limite superioare și inferioare ale marcajelor de timp pentru momentul în care anumite shot-uri au fost executate pe QPU.
Shot-urile sunt grupate în obiecte [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), fiecare indicând un timp de început, un timp de oprire și o specificație a shot-urilor colectate în acel interval.

Un interval de execuție specifică ce date au fost executate în fereastra sa prin furnizarea unei metode [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). Această metodă, pentru orice index [Primitive Unified Block (PUB)](/guides/primitive-input-output), returnează o mască booleană care este `True` pentru toate shot-urile executate în fereastra sa. PUB-urile sunt indexate în ordinea în care au fost furnizate apelului Sampler run. Dacă, de exemplu, un PUB are forma `(2, 3)` și a rulat cu patru shot-uri, atunci forma măștii este `(2, 3, 4)`. Vezi pagina API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) pentru detalii complete.

Exemplu:
Pentru a vizualiza informațiile despre intervalele de execuție, analizează metadatele rezultatului returnat de `SamplerV2`, care iau forma unui obiect `ExecutionSpans`. Acest obiect este un container de tip listă care conține instanțe ale subclaselor lui `ExecutionSpan`, cum ar fi `SliceSpan`: